# Modelo predictivo de recargas — DigiTrans

Este cuaderno explica y reproduce el flujo del modelo predictivo del proyecto. El objetivo es ordenar a los pasajeros activos según su señal estimada de volver a recargar la Billetera durante los próximos 30 días.

**No escribe en Atlas.** Lee una tabla CSV preparada previamente y, al final, permite descargar un CSV local con puntuaciones. No pegues aquí una URI, contraseña ni credenciales de Atlas.

## 1. ¿Qué predice?

- **Unidad:** pasajero y semana de referencia.
- **Objetivo principal:** si se registra una recarga exitosa de Billetera dentro de los próximos 30 días.
- **Cohorte:** usuarios con al menos 2 recargas históricas y 1 recarga durante los últimos 365 días.
- **Señales:** recargas y consumos recientes, viajes, antigüedad, tiempo desde la última actividad y cadencia histórica de recarga.
- **Modelo usado para publicar:** CatBoost binario. Las puntuaciones se convierten en niveles relativos Alto (10%), Medio (30%) y Bajo (60%).

El modelo es una herramienta de priorización. La puntuación no es una promesa de que una persona recargará ni una explicación causal de su comportamiento.

## 2. Flujo de datos del proyecto

1. El m?dulo Modelo exporta usuarios y eventos de la base Atlas del proyecto a CSV.
2. `construir_tabla_usuario_semana.py` crea una fila por usuario y semana. Las variables usan solo informaci?n disponible hasta esa semana; las etiquetas observan el futuro.
3. `evaluar_horizontes.py` mide el rendimiento con ventanas walk-forward m?viles. La publicaci?n del proyecto conserva el horizonte de 30 d?as.


## 3. Preparar la tabla

En tu equipo, desde la carpeta Modelo, genera la tabla con las mismas variables del proyecto:

- exportar_datos_atlas.py produce usuarios_atlas.csv y eventos_atlas.csv. Requiere DIGITRANS_ATLAS_URI en tu entorno local.
- construir_tabla_usuario_semana.py --horizontes 7,14,30 produce tabla_usuario_semana_horizontes_v1.csv.

Sube a Colab únicamente tabla_usuario_semana_horizontes_v1.csv. Si contiene datos reales de usuarios o movimientos, utiliza datos anonimizados y autorizados: el archivo se carga al entorno alojado de Colab. No guardes credenciales en el notebook.

In [ ]:
%pip install -q catboost pandas numpy scikit-learn matplotlib

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display

HORIZONTES = (7, 14, 30)
ANCLA = pd.Timestamp("2024-01-01")
PASO_SEMANAS = 5  # una foto cada 35 d?as para evitar ventanas solapadas
HORIZONTE_PUBLICACION = 30
NUM_CORTES = 6
DIAS_VALIDACION = 90
MIN_CORTES = 3
VERSION_MODELO = "recarga_historial_consumo_transporte_v10_horizonte30_catboost"

VARIABLES_BASE = [
    "rec_7d", "rec_7d_anterior", "rec_30d", "rec_90d", "rec_180d", "rec_365d",
    "tarjeta_7d", "tarjeta_30d", "tarjeta_90d", "tarjeta_180d", "tarjeta_365d",
    "tarjeta_8_30d", "tarjeta_31_90d", "tarjeta_91_180d", "tarjeta_historica",
    "dias_desde_ultima_recarga", "dias_entre_ultimas_recargas", "movil_30d", "movil_90d",
    "viaje_7d", "viaje_30d", "viaje_90d", "viaje_365d", "viaje_historico",
    "dias_desde_ultimo_viaje", "dias_desde_ultima",
    "consumo_7d", "consumo_30d", "consumo_90d", "consumo_365d",
    "gasto_billetera_30d", "gasto_billetera_90d", "gasto_billetera_365d",
    "dias_desde_ultimo_consumo", "antiguedad_dias", "tendencia_7d",
    "servicio_transcaribe", "servicio_transmilenio", "servicio_transmetro",
]
VARIABLES_CADENCIA = [
    "cadencia_mediana_recarga_dias",
    "variabilidad_cadencia_recarga",
    "progreso_ciclo_recarga",
    "intervalos_recarga_observados",
]
VARIABLES = VARIABLES_BASE + VARIABLES_CADENCIA

print(f"Variables predictoras: {len(VARIABLES)}")
print(f"Horizontes evaluados: {HORIZONTES} d?as; horizonte de publicaci?n: {HORIZONTE_PUBLICACION} d?as")
print(f"Validaci?n: hasta {NUM_CORTES} cortes m?viles de {DIAS_VALIDACION} d?as")


## 4. Cargar el CSV en Colab

El archivo debe ser tabla_usuario_semana_horizontes_v1.csv, generado por el script del proyecto. La celda permite seleccionarlo desde tu computador. Para archivos grandes, puedes colocarlo en Google Drive y cambiar RUTA_TABLA por la ruta montada.

In [ ]:
RUTA_TABLA = Path("tabla_usuario_semana_horizontes_v1.csv")

if not RUTA_TABLA.exists():
    from google.colab import files
    print("Selecciona tabla_usuario_semana_horizontes_v1.csv")
    archivos = files.upload()
    if RUTA_TABLA.name not in archivos:
        raise FileNotFoundError("No se subió tabla_usuario_semana_horizontes_v1.csv")

tabla_todas = pd.read_csv(RUTA_TABLA, parse_dates=["semana"])
tabla_todas["semana"] = pd.to_datetime(tabla_todas["semana"], errors="coerce")
tabla_todas = tabla_todas.dropna(subset=["semana"]).copy()

columnas_requeridas = {
    "usuario_id", "semana", "tarjeta_historica", "tarjeta_365d",
    *VARIABLES, *(f"y_{h}d" for h in HORIZONTES)
}
faltantes = sorted(columnas_requeridas - set(tabla_todas.columns))
if faltantes:
    raise ValueError(f"Faltan columnas de la tabla del proyecto: {faltantes}")

columnas_numericas = [
    "tarjeta_historica", "tarjeta_365d", *VARIABLES,
    *(f"y_{h}d" for h in HORIZONTES)
]
for columna in columnas_numericas:
    tabla_todas[columna] = pd.to_numeric(tabla_todas[columna], errors="coerce")

print(f"Filas cargadas: {len(tabla_todas):,}")
print(f"Usuarios distintos: {tabla_todas['usuario_id'].nunique():,}")
print(f"Rango de semanas: {tabla_todas['semana'].min().date()} a {tabla_todas['semana'].max().date()}")

In [ ]:
def pasajero_activo(df):
    return (df["tarjeta_historica"] >= 2) & (df["tarjeta_365d"] >= 1)

def muestra_separada(df):
    semanas = ((df["semana"] - ANCLA).dt.days // 7).astype(int)
    return (semanas % PASO_SEMANAS) == 0

tabla_elegible = tabla_todas[pasajero_activo(tabla_todas)].copy()
tabla_evaluacion = tabla_elegible[muestra_separada(tabla_elegible)].copy()

if not np.isfinite(tabla_evaluacion[VARIABLES].to_numpy(dtype=np.float32)).all():
    raise ValueError("Hay valores vacíos o no numéricos en las variables del modelo.")

resumen_cohorte = pd.DataFrame({
    "Métrica": ["Filas elegibles", "Usuarios elegibles", "Semanas de referencia", "Prevalencia y_30d"],
    "Valor": [
        f"{len(tabla_elegible):,}",
        f"{tabla_elegible['usuario_id'].nunique():,}",
        f"{tabla_elegible['semana'].nunique():,}",
        f"{tabla_elegible['y_30d'].dropna().mean():.1%}",
    ],
})
display(resumen_cohorte)

## 5. Preparación y configuración del modelo

El escalado reproduce modelo_predictivo_config.py: aplica log1p a conteos, montos y recencias para reducir asimetría; mantiene los indicadores de servicio binarios y transforma tendencia_7d conservando el signo.

CatBoost utiliza pesos automáticos para compensar el desbalance de clases. La configuración no requiere GPU.

In [ ]:
def transformar_variables(valores, variables=VARIABLES):
    x = np.asarray(valores, dtype=np.float32).copy()
    indice_tendencia = variables.index("tendencia_7d")
    indicadores = {
        variables.index("servicio_transcaribe"),
        variables.index("servicio_transmilenio"),
        variables.index("servicio_transmetro"),
    }
    indices_log = [
        i for i in range(x.shape[1])
        if i != indice_tendencia and i not in indicadores
    ]
    x[:, indices_log] = np.log1p(np.maximum(x[:, indices_log], 0))
    x[:, indice_tendencia] = (
        np.sign(x[:, indice_tendencia]) * np.log1p(np.abs(x[:, indice_tendencia]))
    )
    return x

def crear_modelo():
    return CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="SqrtBalanced",
        l2_leaf_reg=8.0,
        random_seed=42,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
    )

def precision_en_top(y, puntuaciones, proporcion):
    n = max(1, int(np.ceil(len(y) * proporcion)))
    mejores = np.argsort(-puntuaciones, kind="mergesort")[:n]
    positivos = int(np.asarray(y)[mejores].sum())
    total_positivos = int(np.asarray(y).sum())
    precision = positivos / n
    recall = positivos / total_positivos if total_positivos else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1, n, positivos

## 6. Evaluaci?n temporal walk-forward

La evaluaci?n crea hasta seis ventanas recientes, cada una de 90 d?as, a partir de la ?ltima etiqueta completa. Entrena con periodos anteriores, deja un embargo del tama?o del horizonte y exige al menos tres cortes con clases positivas y negativas.

Las ventanas nuevas permiten medir los datos hist?ricos m?s recientes al regenerar la tabla. AUC-PR ayuda con clases desbalanceadas; precisi?n y recall del top 10% describen una pol?tica de priorizaci?n, no una probabilidad garantizada para cada persona.

Se siguen mostrando 7 y 14 d?as como referencia de evaluaci?n; la publicaci?n y el objetivo elegido del proyecto contin?an en 30 d?as.


In [ ]:
def construir_cortes_temporales(df, horizonte):
    objetivo = f"y_{horizonte}d"
    etiquetadas = df[df[objetivo].notna()].copy()
    if etiquetadas.empty:
        raise ValueError(f"No hay etiquetas completas para {horizonte} d?as.")

    ultima_semana = etiquetadas["semana"].max()
    ancho = pd.Timedelta(days=DIAS_VALIDACION)
    cortes = []
    for desplazamiento in range(NUM_CORTES):
        fin_valid = ultima_semana - desplazamiento * ancho
        inicio_valid = fin_valid - pd.Timedelta(days=DIAS_VALIDACION - 1)
        fin_train = inicio_valid - pd.Timedelta(days=horizonte + 1)
        train = etiquetadas[etiquetadas["semana"] <= fin_train]
        valid = etiquetadas[
            (etiquetadas["semana"] >= inicio_valid)
            & (etiquetadas["semana"] <= fin_valid)
        ]
        if train.empty or valid.empty:
            continue
        if train["semana"].max() + pd.Timedelta(days=horizonte) >= valid["semana"].min():
            raise ValueError(f"El embargo temporal es insuficiente para {horizonte} d?as.")
        cortes.append((f"corte_{NUM_CORTES - desplazamiento:02d}", train, valid))

    cortes.reverse()
    if len(cortes) < MIN_CORTES:
        raise ValueError(
            f"Solo hay {len(cortes)} cortes completos para {horizonte} d?as; "
            f"se necesitan al menos {MIN_CORTES}."
        )
    return cortes

filas_metricas = []
modelo_30d_ultimo_corte = None

for horizonte in HORIZONTES:
    objetivo = f"y_{horizonte}d"
    cortes = construir_cortes_temporales(tabla_evaluacion, horizonte)
    for nombre_corte, train, valid in cortes:
        y_train = train[objetivo].to_numpy(dtype=np.int8)
        y_valid = valid[objetivo].to_numpy(dtype=np.int8)
        if np.unique(y_train).size < 2 or np.unique(y_valid).size < 2:
            raise ValueError(f"{nombre_corte}, {horizonte} d?as: faltan ejemplos de ambas clases.")

        x_train = transformar_variables(train[VARIABLES].to_numpy(dtype=np.float32))
        x_valid = transformar_variables(valid[VARIABLES].to_numpy(dtype=np.float32))
        modelo = crear_modelo()
        modelo.fit(x_train, y_train)
        puntuaciones = modelo.predict_proba(x_valid)[:, 1]

        p5, r5, f5, n5, pos5 = precision_en_top(y_valid, puntuaciones, 0.05)
        p10, r10, f10, n10, pos10 = precision_en_top(y_valid, puntuaciones, 0.10)
        p20, r20, f20, n20, pos20 = precision_en_top(y_valid, puntuaciones, 0.20)
        filas_metricas.append({
            "corte": nombre_corte,
            "horizonte_dias": horizonte,
            "filas_entrenamiento": len(train),
            "filas_validacion": len(valid),
            "inicio_validacion": valid["semana"].min().strftime("%Y-%m-%d"),
            "fin_validacion": valid["semana"].max().strftime("%Y-%m-%d"),
            "prevalencia": float(y_valid.mean()),
            "auc_roc": roc_auc_score(y_valid, puntuaciones),
            "auc_pr": average_precision_score(y_valid, puntuaciones),
            "precision_top_5": p5,
            "recall_top_5": r5,
            "f1_top_5": f5,
            "precision_top_10": p10,
            "recall_top_10": r10,
            "f1_top_10": f10,
            "precision_top_20": p20,
            "recall_top_20": r20,
            "f1_top_20": f20,
        })
        if horizonte == HORIZONTE_PUBLICACION:
            modelo_30d_ultimo_corte = modelo
        print(
            f"{nombre_corte} | {horizonte:>2} d?as | "
            f"validaci?n {valid['semana'].min().date()} a {valid['semana'].max().date()} | "
            f"AUC-PR={average_precision_score(y_valid, puntuaciones):.3f} | "
            f"precisi?n top 10%={p10:.1%} | recall top 10%={r10:.1%}"
        )

resultados = pd.DataFrame(filas_metricas)
display(resultados.round(4))


In [ ]:
resumen_horizontes = resultados.groupby("horizonte_dias", as_index=False).agg(
    auc_pr_media=("auc_pr", "mean"),
    prevalencia_media=("prevalencia", "mean"),
    precision_top_10_media=("precision_top_10", "mean"),
    recall_top_10_media=("recall_top_10", "mean"),
)
display(resumen_horizontes.round(4))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(resumen_horizontes["horizonte_dias"], resumen_horizontes["auc_pr_media"], marker="o")
ejes[0].set(title="AUC-PR promedio por horizonte", xlabel="Días", ylabel="AUC-PR")
ejes[0].grid(alpha=0.25)
ejes[1].plot(resumen_horizontes["horizonte_dias"], resumen_horizontes["precision_top_10_media"], marker="o", label="Precisión top 10%")
ejes[1].plot(resumen_horizontes["horizonte_dias"], resumen_horizontes["prevalencia_media"], marker="s", linestyle="--", label="Prevalencia")
ejes[1].set(title="Segmento superior frente a la base", xlabel="Días", ylabel="Proporción")
ejes[1].legend()
ejes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. ¿Qué variables pesan más en el último corte mensual?

La importancia de CatBoost mide cuánto contribuyó una variable a las divisiones del modelo en ese entrenamiento. No muestra causalidad, y el orden puede variar si cambia la muestra.

In [ ]:
importancias = pd.DataFrame({
    "variable": VARIABLES,
    "importancia": modelo_30d_ultimo_corte.feature_importances_,
}).sort_values("importancia", ascending=False).head(20)

display(importancias.head(10).round(3))

plt.figure(figsize=(9, 6))
plt.barh(importancias["variable"][::-1], importancias["importancia"][::-1], color="#7250d5")
plt.title("20 variables más importantes — modelo de 30 días")
plt.xlabel("Importancia CatBoost")
plt.tight_layout()
plt.show()

## 8. Entrenar con el historial maduro y puntuar la semana más reciente

Para publicar, el proyecto reserva las etiquetas de 30 días que todavía no han madurado, entrena con el historial elegible y puntúa a todos los pasajeros activos de la última semana. Los niveles son relativos al grupo puntuado: 10% Alto, 30% Medio y 60% Bajo.

El cuaderno solo crea resultados en esta sesión de Colab. No se conecta a Atlas.

In [ ]:
semana_referencia = tabla_todas["semana"].max()
limite_entrenamiento = semana_referencia - pd.Timedelta(days=HORIZONTE_PUBLICACION)

entrenamiento_final = tabla_todas[
    (tabla_todas["semana"] >= ANCLA)
    & (tabla_todas["semana"] < limite_entrenamiento)
    & tabla_todas["y_30d"].notna()
    & pasajero_activo(tabla_todas)
    & muestra_separada(tabla_todas)
].copy()

actual = tabla_todas[
    (tabla_todas["semana"] == semana_referencia)
    & pasajero_activo(tabla_todas)
].copy()

if entrenamiento_final.empty or entrenamiento_final["y_30d"].nunique() < 2:
    raise ValueError("No hay etiquetas maduras suficientes para entrenar.")
if actual.empty:
    raise ValueError("No hay pasajeros activos en la semana de referencia más reciente.")

x_train_final = transformar_variables(
    entrenamiento_final[VARIABLES].to_numpy(dtype=np.float32)
)
x_actual = transformar_variables(actual[VARIABLES].to_numpy(dtype=np.float32))
if not np.isfinite(x_train_final).all() or not np.isfinite(x_actual).all():
    raise ValueError("Hay valores vacíos o no numéricos en las variables.")

modelo_final = crear_modelo()
modelo_final.fit(x_train_final, entrenamiento_final["y_30d"].to_numpy(dtype=np.int8))
scores = modelo_final.predict_proba(x_actual)[:, 1]

corte_alto = np.quantile(scores, 0.90)
corte_medio = np.quantile(scores, 0.60)
niveles = np.where(scores >= corte_alto, "Alto",
          np.where(scores >= corte_medio, "Medio", "Bajo"))
rango = pd.Series(scores).rank(method="first")
deciles = pd.qcut(rango, 10, labels=False, duplicates="drop").astype(int).to_numpy() + 1

predicciones = pd.DataFrame({
    "usuario_id": actual["usuario_id"].astype(str).to_numpy(),
    "nivel_riesgo": niveles,
    "decil": deciles,
    "score_modelo": scores,
    "semana_referencia": semana_referencia.strftime("%Y-%m-%d"),
    "objetivo": "recarga_billetera_en_30_dias",
    "horizonte_dias": HORIZONTE_PUBLICACION,
    "version_modelo": VERSION_MODELO,
})

resumen_segmentos = predicciones.groupby("nivel_riesgo", as_index=False).agg(
    pasajeros=("usuario_id", "count"),
    score_medio=("score_modelo", "mean"),
)
orden = pd.Categorical(resumen_segmentos["nivel_riesgo"], ["Alto", "Medio", "Bajo"], ordered=True)
resumen_segmentos = resumen_segmentos.assign(orden=orden).sort_values("orden").drop(columns="orden")

print(f"Semana de referencia: {semana_referencia.date()}")
print(f"Filas para entrenamiento: {len(entrenamiento_final):,}")
print(f"Pasajeros activos puntuados: {len(predicciones):,}")
display(resumen_segmentos.round(4))

## 9. Descargar el resultado

Este archivo contiene identificadores de usuario. Descárgalo y guárdalo únicamente en un lugar autorizado. No lo compartas públicamente.

In [ ]:
ARCHIVO_SALIDA = "predicciones_recarga_colab.csv"
predicciones.drop(columns=["score_modelo"]).to_csv(ARCHIVO_SALIDA, index=False)

from google.colab import files
files.download(ARCHIVO_SALIDA)

## 10. Cómo leer los resultados

- **AUC-PR:** resume el ordenamiento de positivos frente a negativos y conviene compararlo con la prevalencia.
- **Precisión top 10%:** proporción de pasajeros que sí recargaron dentro del segmento con mayor puntuación.
- **Recall top 10%:** proporción de todas las recargas observadas que quedó dentro de ese segmento.
- **Alto/Medio/Bajo:** grupos de tamaño relativo; no son probabilidades calibradas ni niveles absolutos.
- **Importancia de variables:** asociación dentro del modelo, no demuestra que una variable cause una recarga.

Los datos actuales del proyecto pueden ser sintéticos. Si lo son, las métricas sirven para revisar el flujo de trabajo y no representan el rendimiento que se obtendría con pasajeros reales.